# LSTM Reimplementation

This notebook reproduces the LSTM-based fake news detection model from:

> Liu, S. (2023). *Social Media Fake Information Identification Method Based on LSTM*. GEFHR 2023, Vol. 21, pp. 703–709.

All model parameters follow the original paper exactly (Table 2). The goal is to verify that the reported 99% accuracy and 100% AUC are reproducible, and to understand what drives those numbers given the dataset bias identified in `00_eda.ipynb`.

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix, ConfusionMatrixDisplay

# Keras is now integrated into TensorFlow (tf.keras = Keras)
# keras.preprocessing.text was removed in Keras 3 — use tensorflow.keras instead
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

os.makedirs('../results', exist_ok=True)
print('TensorFlow:', tf.__version__)
print('Keras:', tf.keras.__version__)

## 2. Load Data

In [ ]:
fake = pd.read_csv('../data/raw/Fake.csv')
true = pd.read_csv('../data/raw/True.csv')

fake['label'] = 0
true['label'] = 1

df = pd.concat([fake, true]).sample(frac=1, random_state=42).reset_index(drop=True)

print(f'Total: {len(df):,} rows')
print(df['label'].value_counts())

原论文使用 `title + text` 作为输入文本，`subject` 和 `date` 列并未显式排除，因此模型可能接触到了泄露特征（见 `00_eda.ipynb`）。为忠实复现原论文，这里沿用相同设置——使用 `text` 列作为输入。

The original paper uses the article text as input without explicitly removing `subject` or `date`. To faithfully reproduce the paper's results, we follow the same setup here and use the `text` column as input.